<a href="https://colab.research.google.com/github/frank-morales2020/Cloud_curious/blob/master/DeepSeek_R1_Distill_Llama_70B_DRIVIA_H2E_Multimodal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Download

In [ ]:
import os

# Must be set before ALL imports to prevent XET local disk buffering
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_CACHE"]       = "/content/drive/MyDrive/H2E_Models/.cache"
os.environ["HF_HOME"]            = "/content/drive/MyDrive/H2E_Models/.cache"

from google.colab import drive, userdata
from huggingface_hub import snapshot_download

# =============================================================================
# 1. SOVEREIGN AUTHENTICATION & PATHS
# =============================================================================
drive.mount('/content/drive', force_remount=True)

MODEL_BASE = "/content/drive/MyDrive/H2E_Models"
VJ_PATH    = os.path.join(MODEL_BASE, "v_jepa")
IJ_PATH    = os.path.join(MODEL_BASE, "i_jepa")
DS_PATH    = os.path.join(MODEL_BASE, "deepseek_v3")


# Create all folders BEFORE any download or os.listdir call
os.makedirs(VJ_PATH,               exist_ok=True)
os.makedirs(IJ_PATH,               exist_ok=True)
os.makedirs(DS_PATH,               exist_ok=True)
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

HF_TOKEN = userdata.get('HF_TOKEN')

# =============================================================================
# 2. PERCEPTUAL INPUTS (THE EYES) — small enough for snapshot_download
# =============================================================================
print("[*] Downloading V-JEPA 2 (Temporal Perception)...")
snapshot_download(
    repo_id="facebook/vjepa2-vitl-fpc64-256",
    local_dir=VJ_PATH,
    token=HF_TOKEN,
    local_dir_use_symlinks=False
)

print("[*] Downloading I-JEPA (Spatial Perception)...")
snapshot_download(
    repo_id="facebook/ijepa_vith14_1k",
    local_dir=IJ_PATH,
    token=HF_TOKEN,
    local_dir_use_symlinks=False
)

# =============================================================================
# 3. COGNITIVE LAYER (THE BRAIN)
# wget streams each shard DIRECTLY to Drive — zero local VM disk used
# =============================================================================
print("[*] Fetching DeepSeek V3 Q4_K_M shard list from HuggingFace...")

import requests

API_URL = "https://huggingface.co/api/models/unsloth/DeepSeek-V3-GGUF"
headers = {"Authorization": f"Bearer {HF_TOKEN}"}
resp    = requests.get(API_URL, headers=headers)
files   = [
    s["rfilename"]
    for s in resp.json().get("siblings", [])
    if "Q4_K_M" in s["rfilename"] and s["rfilename"].endswith(".gguf")
]
files.sort()

print(f"  Found {len(files)} Q4_K_M shards:")
for f in files:
    print(f"    • {f}")

BASE_URL = "https://huggingface.co/unsloth/DeepSeek-V3-GGUF/resolve/main"

print("\n[*] Downloading DeepSeek V3 Q4_K_M shards directly to Drive...")
for fname in files:
    # fname includes subfolder e.g. DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00001-of-00009.gguf
    dest_dir = os.path.join(DS_PATH, os.path.dirname(fname))
    dest     = os.path.join(DS_PATH, fname)
    os.makedirs(dest_dir, exist_ok=True)

    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f"  ⏭️  Already exists, skipping: {fname}")
        continue

    url = f"{BASE_URL}/{fname}"
    print(f"\n  ⬇️  {fname}")
    # -c = resume if interrupted, -q = quiet, --show-progress = progress bar
    ret = os.system(
        f'wget --header="Authorization: Bearer {HF_TOKEN}" '
        f'-c -q --show-progress '
        f'-O "{dest}" '
        f'"{url}"'
    )
    if ret == 0:
        print(f"  ✅  Saved → {dest}")
    else:
        print(f"  ❌  wget failed for {fname} (exit code {ret})")

# =============================================================================
# 4. SOVEREIGN INTEGRITY VERIFICATION
# =============================================================================
print("\n--- SOVEREIGN REPOSITORY STATUS ---")

def _safe_has_files(path, extensions):
    """Recursively check for any file with given extensions. Never crashes."""
    if not os.path.exists(path):
        return False
    for root, _, files in os.walk(path):
        for f in files:
            if f.startswith("."):
                continue
            if any(f.endswith(ext) for ext in extensions):
                print(f"    📦  {os.path.join(root, f)}")
                return True
    return False

def _safe_find_gguf(path):
    """Recursively find any .gguf file under path. Never crashes."""
    if not os.path.exists(path):
        return False
    for root, _, files in os.walk(path):
        for f in sorted(files):
            if f.endswith(".gguf"):
                print(f"    📦  {os.path.join(root, f)}")
                return True
    return False

print("  Checking V-JEPA 2...")
vj_check = _safe_has_files(VJ_PATH, [".pth", ".bin", ".safetensors", ".pt"])

print("  Checking I-JEPA...")
ij_check = _safe_has_files(IJ_PATH, [".pth", ".bin", ".safetensors", ".pt"])

print("  Checking DeepSeek V3...")
ds_check = _safe_find_gguf(DS_PATH)

status = {
    "V-JEPA 2 (Temporal)": vj_check,
    "I-JEPA (Spatial)":    ij_check,
    "DeepSeek V3 (Brain)": ds_check,
}

print()
for name, secured in status.items():
    icon = "✅" if secured else "❌"
    print(f"  {icon} {name}")

if all(status.values()):
    print("\n🚀 H2E-JEPA v4 ALL MODELS DOWNLOADED. READY TO RUN DEMO.")
else:
    print("\n⚠️  Some components failed. Check HF_TOKEN permissions and Drive space.")


Mounted at /content/drive
[*] Downloading V-JEPA 2 (Temporal Perception)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

[*] Downloading I-JEPA (Spatial Perception)...


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

[*] Fetching DeepSeek V3 Q4_K_M shard list from HuggingFace...
  Found 9 Q4_K_M shards:
    • DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00001-of-00009.gguf
    • DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00002-of-00009.gguf
    • DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00003-of-00009.gguf
    • DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00004-of-00009.gguf
    • DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00005-of-00009.gguf
    • DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00006-of-00009.gguf
    • DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00007-of-00009.gguf
    • DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00008-of-00009.gguf
    • DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00009-of-00009.gguf

[*] Downloading DeepSeek V3 Q4_K_M shards directly to Drive...
  ⏭️  Already exists, skipping: DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00001-of-00009.gguf
  ⏭️  Already exists, skipping: DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00002-of-00009.gguf
  ⏭️  Already exists, skipping: DeepSeek-V3-Q4_K_M/DeepSeek-V3-Q4_K_M-00003-of-00009.gguf
  

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from google.colab import userdata
from huggingface_hub import hf_hub_download
import os

from warnings import filterwarnings
filterwarnings("ignore")

# 1. AUTHENTICATION
HF_TOKEN = userdata.get('HF_TOKEN')

# 2. DEFINE REPO AND FILE
# Using bartowski's repo for a reliable Q4_K_M quant
REPO_ID = "bartowski/DeepSeek-R1-Distill-Llama-70B-GGUF"
FILENAME = "DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf"
DEST_PATH = "/content/drive/MyDrive/H2E_Models/deepseek_70b"

os.makedirs(DEST_PATH, exist_ok=True)

# 3. SECURE DOWNLOAD
print(f"[*] Downloading {FILENAME} to Drive...")
path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    local_dir=DEST_PATH,
    token=HF_TOKEN,
    local_dir_use_symlinks=False,
    resume_download=True  # Important for a 43GB file
)

print(f"[*] Securely saved to: {path}")

[*] Downloading DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf to Drive...


DeepSeek-R1-Distill-Llama-70B-Q4_K_M.ggu(…):   0%|          | 0.00/42.5G [00:00<?, ?B/s]

[*] Securely saved to: /content/drive/MyDrive/H2E_Models/deepseek_70b/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf


## Runtime - DEMO

CELL1

In [3]:
!nvidia-smi

Wed Apr 22 23:05:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             45W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [4]:
# ============================================================
# CELL 1 — Run this FIRST, before any other cell
# Mounts Drive and redirects ALL local cache to Drive
# ============================================================
import os, shutil
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DRIVE_BASE        = "/content/drive/MyDrive/H2E_Models"
DRIVE_TMP         = os.path.join(DRIVE_BASE, ".tmp")
DRIVE_CACHE       = os.path.join(DRIVE_BASE, ".cache")
DRIVE_HF_CACHE    = os.path.join(DRIVE_BASE, ".hf_cache")
DRIVE_TORCH_CACHE = os.path.join(DRIVE_BASE, ".torch_cache")
DRIVE_LLAMA_CACHE = os.path.join(DRIVE_BASE, ".llama_cache")
DRIVE_TRITON      = os.path.join(DRIVE_TORCH_CACHE, ".triton")
DRIVE_PIP         = os.path.join(DRIVE_CACHE, ".pip")

# Create all Drive target dirs
for d in [DRIVE_TMP, DRIVE_CACHE, DRIVE_HF_CACHE, DRIVE_TORCH_CACHE,
          DRIVE_LLAMA_CACHE, DRIVE_TRITON, DRIVE_PIP]:
    os.makedirs(d, exist_ok=True)

# Every path that any C library, Python lib, or Colab kernel
# writes to by default — symlinked to Drive before any import
SYMLINKS = {
    "/root/.cache/huggingface": DRIVE_HF_CACHE,
    "/root/.cache/torch":       DRIVE_TORCH_CACHE,
    "/root/.cache/llama.cpp":   DRIVE_LLAMA_CACHE,
    "/root/.cache/pip":         DRIVE_PIP,
    "/root/.triton":            DRIVE_TRITON,
    "/root/.cache/matplotlib":  os.path.join(DRIVE_CACHE, ".matplotlib"),
    "/root/.cache/jax":         os.path.join(DRIVE_CACHE, ".jax"),
    "/tmp":                     DRIVE_TMP,
}

for local, target in SYMLINKS.items():
    os.makedirs(target, exist_ok=True)
    parent = os.path.dirname(local)
    if parent:
        os.makedirs(parent, exist_ok=True)
    # Remove whatever is there — dir, file, or stale symlink
    if os.path.islink(local):
        if os.readlink(local) == target:
            continue          # already correct
        os.remove(local)
    elif os.path.isdir(local):
        shutil.rmtree(local)
    elif os.path.exists(local):
        os.remove(local)
    os.symlink(target, local)
    print(f"  ✅ {local} → {target}")

# Set env vars as a second layer of defense
os.environ.update({
    "HOME":                   DRIVE_BASE,
    "TMPDIR":                 DRIVE_TMP,
    "TEMP":                   DRIVE_TMP,
    "TMP":                    DRIVE_TMP,
    "XDG_CACHE_HOME":         DRIVE_CACHE,
    "XDG_CONFIG_HOME":        DRIVE_CACHE,
    "XDG_DATA_HOME":          DRIVE_CACHE,
    "HF_HOME":                DRIVE_HF_CACHE,
    "HF_HUB_CACHE":           DRIVE_HF_CACHE,
    "TRANSFORMERS_CACHE":     DRIVE_HF_CACHE,
    "HUGGINGFACE_HUB_CACHE":  DRIVE_HF_CACHE,
    "TORCH_HOME":             DRIVE_TORCH_CACHE,
    "TORCH_EXTENSIONS_DIR":   os.path.join(DRIVE_TORCH_CACHE, "extensions"),
    "TRITON_CACHE_DIR":       DRIVE_TRITON,
    "TORCHINDUCTOR_CACHE_DIR":os.path.join(DRIVE_TORCH_CACHE, ".inductor"),
    "LLAMA_CACHE":            DRIVE_LLAMA_CACHE,
    "LLAMA_CPP_CACHE":        DRIVE_LLAMA_CACHE,
    "LLAMA_CPP_TEMP":         DRIVE_TMP,
    "PYTHON_EGG_CACHE":       DRIVE_TMP,
    "PYTHONPYCACHEPREFIX":    DRIVE_TMP,
    "JAX_PLATFORM_NAME":      "cuda",
    "XLA_FLAGS":              "--xla_gpu_deterministic_ops=true",
    "LLAMA_CPP_VERBOSE":      "0",
    "LLAMA_CPP_LOGGING":      "0",
})

import tempfile
tempfile.tempdir = DRIVE_TMP

print("\n✅ All cache/temp → Drive. Now run Cell 2 (setup) then Cell 3 (main script).")


Mounted at /content/drive
  ✅ /root/.cache/huggingface → /content/drive/MyDrive/H2E_Models/.hf_cache
  ✅ /root/.cache/torch → /content/drive/MyDrive/H2E_Models/.torch_cache
  ✅ /root/.cache/llama.cpp → /content/drive/MyDrive/H2E_Models/.llama_cache
  ✅ /root/.cache/pip → /content/drive/MyDrive/H2E_Models/.cache/.pip
  ✅ /root/.triton → /content/drive/MyDrive/H2E_Models/.torch_cache/.triton
  ✅ /root/.cache/matplotlib → /content/drive/MyDrive/H2E_Models/.cache/.matplotlib
  ✅ /root/.cache/jax → /content/drive/MyDrive/H2E_Models/.cache/.jax
  ✅ /tmp → /content/drive/MyDrive/H2E_Models/.tmp

✅ All cache/temp → Drive. Now run Cell 2 (setup) then Cell 3 (main script).


CELL 2

In [5]:
"""
H2E-JEPA — Cell 2: package installs
=====================================
Run once per session, after h2e_cell1.py.
A runtime restart may occur after JAX installs — that is expected.
Re-run this cell after the restart, then run h2e_cell3.py.
"""

import subprocess, sys

pkgs = [
    # core
    "tqdm",
    "numpy",
    "flax",
    "orbax-checkpoint",
    # vision
    "torch torchvision --index-url https://download.pytorch.org/whl/cu121",
    "Pillow",
    "opencv-python-headless",
    "transformers accelerate",
    # llama
    "llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121",
    # audio (optional)
    "moviepy",
    "openai-whisper",
]

for pkg in pkgs:
    print(f"  📦 Installing {pkg.split()[0]}...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + pkg.split(),
        check=False,
    )

# Flash Attention — prebuilt wheel for cu12 + torch2.10 + cp312
print("  📦 Installing flash-attention...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "https://github.com/lesj0610/flash-attention/releases/download/"
     "v2.8.3-cu12-torch2.10-cp312/"
     "flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"],
    check=False,
)

# Validate flash attention — fall back to SDPA if unavailable
attn_impl = "sdpa"
try:
    import flash_attn
    attn_impl = "flash_attention_2"
    print(f"  ✅ Flash Attention {flash_attn.__version__} validated")
except Exception:
    print("  ⚠️  Flash Attention unavailable — falling back to SDPA")

print(f"  ℹ️  attn_impl = {attn_impl!r}")

# JAX CUDA — must come last, may trigger runtime restart
print("  📦 Installing jax[cuda12]...")
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y",
     "nvidia-cuda-nvcc", "nvidia-cuda-nvcc-cu12"],
    capture_output=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "jax[cuda12]",
     "-f", "https://storage.googleapis.com/jax-releases/jax_cuda_releases.html"],
    check=False,
)

print("\n✅ Setup complete — if the runtime restarts, run this cell again then run h2e_cell3.py.")


  📦 Installing tqdm...
  📦 Installing numpy...
  📦 Installing flax...
  📦 Installing orbax-checkpoint...
  📦 Installing torch...
  📦 Installing Pillow...
  📦 Installing opencv-python-headless...
  📦 Installing transformers...
  📦 Installing llama-cpp-python...
  📦 Installing moviepy...
  📦 Installing openai-whisper...
  📦 Installing flash-attention...
  ✅ Flash Attention 2.8.3 validated
  ℹ️  attn_impl = 'flash_attention_2'
  📦 Installing jax[cuda12]...

✅ Setup complete — if the runtime restarts, run this cell again then run h2e_cell3.py.


CELL 3

In [ ]:
"""
H2E-JEPA v4 — Human-to-Expert Perceptual Socratic Tutor
========================================================
"""

import os
import time
import threading
import warnings
import random
import json
import uuid
import datetime
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Generator, Iterator, List, Optional, Tuple

import numpy as np
import jax
import jax.numpy as jnp
from flax import linen as nn
from tqdm.auto import tqdm

# Suppress warnings
warnings.filterwarnings("ignore")

# Drive paths
DRIVE_BASE = "/content/drive/MyDrive/H2E_Models"
DRIVE_TMP = os.path.join(DRIVE_BASE, ".tmp")
DRIVE_HF_CACHE = os.path.join(DRIVE_BASE, ".hf_cache")
DRIVE_LLAMA_CACHE = os.path.join(DRIVE_BASE, ".llama_cache")

# Check redirect
assert os.path.islink("/tmp"), "❌ /tmp is not a symlink — run h2e_cell1_redirect.py first"
print("✅ Drive redirect confirmed")

# PyTorch imports
try:
    import torch
    import torch.nn.functional as F
    from PIL import Image
    import torchvision.transforms as T
    from transformers import AutoModel, AutoProcessor, AutoVideoProcessor
    _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False

# Llama-cpp
try:
    from llama_cpp import Llama as _Llama
    _HAS_LLAMA = True
except ImportError:
    _HAS_LLAMA = False

# Audio
try:
    from moviepy.editor import VideoFileClip
    import whisper as _whisper
    _HAS_AUDIO = True
except ImportError:
    _HAS_AUDIO = False

# =============================================================================
# CONFIGURATION
# =============================================================================
SEED = 123
SROI_THRESHOLD = 0.50
SROI_FLOOR = 0.15
VJEPA_EMBED_DIM = 1024
ENCODER_DEPTH = 4
ENCODER_HEADS = 8
ENCODER_FF_MULT = 4
ENCODER_DROPOUT = 0.1
NUM_FRAMES = 16
ATTN_IMPL = "flash_attention_2"
AUDIO_EMBED_DIM = 512
FUSED_DIM_VISUAL = 1280 + 1024
FUSED_DIM_AUDIO = 1280 + 1024 + AUDIO_EMBED_DIM

VIDEO_PATH = "/content/drive/MyDrive/datasets/TartanAviation/vision/1_2023-02-22-15-21-49/1_2023-02-22-15-21-49.mp4"
MODEL_BASE = "/content/drive/MyDrive/H2E_Models"
IJEPA_PATH = os.path.join(MODEL_BASE, "i_jepa")
VJEPA_PATH = os.path.join(MODEL_BASE, "v_jepa")
DS_PATH = os.path.join(MODEL_BASE, "deepseek_70b")
SESSION_DIR = Path(DRIVE_BASE) / "sessions"
SESSION_DIR.mkdir(parents=True, exist_ok=True)

def _find_gguf(path: str) -> Optional[str]:
    if not os.path.exists(path):
        return None
    for root, _, files in os.walk(path):
        for f in sorted(files):
            if f.endswith(".gguf"):
                return os.path.join(root, f)
    return None

def _get_total_size(path: str) -> Tuple[float, int]:
    if not os.path.exists(path):
        return 0.0, 0
    total, count = 0, 0
    for root, _, files in os.walk(path):
        for f in files:
            if f.endswith(".gguf"):
                total += os.path.getsize(os.path.join(root, f))
                count += 1
    return total / (1024 ** 3), count

DEEPSEEK_GGUF = _find_gguf(DS_PATH)
_total_size_gb, _shard_count = _get_total_size(DS_PATH)

def _lock_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if _HAS_TORCH and torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

_lock_seed()
_h2e_key = jax.random.PRNGKey(SEED)
_enc_key = jax.random.PRNGKey(SEED + 1)

# =============================================================================
# TQDM UTILITY
# =============================================================================
_DRIVE_BYTES_PER_SEC = 60 * 1024 * 1024

def _make_tqdm_loader(total_bytes: int, desc: str):
    pbar = tqdm(total=total_bytes, desc=desc, unit="B", unit_scale=True)
    stop = threading.Event()
    def _advance():
        step = int(_DRIVE_BYTES_PER_SEC * 0.25)
        while not stop.is_set():
            increment = min(step, total_bytes - pbar.n)
            if increment > 0:
                pbar.update(increment)
            stop.wait(0.25)
    t = threading.Thread(target=_advance, daemon=True)
    t.start()
    return pbar, stop, t

# =============================================================================
# VIDEO LOADER
# =============================================================================
class TartanVideoLoader:
    def load(self, video_path: str = None, num_frames: int = NUM_FRAMES) -> Tuple[List, str]:
        if video_path and os.path.exists(video_path):
            frames = self._from_path(video_path, num_frames)
            if frames:
                print(f"  🎬 Loaded {len(frames)} frames")
                return frames, video_path
        print("  ⚠️  Video not found — using synthetic frames")
        return self._synthetic(num_frames), "synthetic"

    def _from_path(self, path: str, num_frames: int) -> List:
        try:
            import cv2
        except ImportError:
            return []
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0:
            cap.release()
            return []
        target_indices = set(np.linspace(0, total - 1, min(num_frames, total), dtype=int).tolist())
        frames, idx = [], 0
        while cap.isOpened() and len(frames) < num_frames:
            ret, frame = cap.read()
            if not ret:
                break
            if idx in target_indices:
                frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
            idx += 1
        cap.release()
        return frames

    def _synthetic(self, num_frames: int) -> List:
        rng = np.random.default_rng(SEED)
        frames = []
        for i in range(num_frames):
            base = rng.integers(60, 160, (32, 32, 3), dtype=np.uint8)
            detail = rng.integers(0, 40, (256, 256, 3), dtype=np.uint8)
            coarse = np.array(Image.fromarray(base).resize((256, 256), Image.NEAREST))
            drift = int(40 * np.sin(i * np.pi / num_frames))
            frame = np.clip(coarse.astype(np.int16) + detail + drift - 20, 0, 255).astype(np.uint8)
            frames.append(Image.fromarray(frame))
        return frames

    def extract_keyframe(self, frames: List) -> Optional[Image.Image]:
        return frames[len(frames) // 2] if frames else None

    def extract_audio_embed(self, video_path: str) -> Tuple[Optional[torch.Tensor], str]:
        if not _HAS_AUDIO:
            return None, ""
        if not os.path.exists(video_path):
            return None, ""
        try:
            clip = VideoFileClip(video_path)
            if clip.audio is None:
                clip.close()
                return None, ""
            wav_path = os.path.join(DRIVE_TMP, f"tmp_audio_{uuid.uuid4().hex[:8]}.wav")
            clip.audio.write_audiofile(wav_path, codec='pcm_s16le', fps=16000, verbose=False, logger=None)
            clip.close()
            if not os.path.exists(wav_path):
                return None, ""
            model = _whisper.load_model("base", download_root=DRIVE_HF_CACHE)
            result = model.transcribe(wav_path, fp16=torch.cuda.is_available(), verbose=False)
            transcript = result.get("text", "")
            audio = _whisper.load_audio(wav_path)
            audio = _whisper.pad_or_trim(audio)
            mel = _whisper.log_mel_spectrogram(audio).to(model.device)
            with torch.no_grad():
                emb = model.encoder(mel.unsqueeze(0)).mean(dim=1)
            emb = F.normalize(emb.float(), dim=-1).cpu()
            os.remove(wav_path)
            return emb, transcript
        except Exception:
            return None, ""

# =============================================================================
# PERCEPTUAL MODELS
# =============================================================================
class IJEPAExtractor:
    _MEAN = (0.485, 0.456, 0.406)
    _STD = (0.229, 0.224, 0.225)
    _SIZE = 224

    def __init__(self):
        self._model = None
        self._processor = None

    def _load(self):
        if self._model is not None:
            return
        print("  📷 Loading I-JEPA...")
        try:
            self._processor = AutoProcessor.from_pretrained(IJEPA_PATH, cache_dir=DRIVE_HF_CACHE)
        except Exception:
            self._processor = None
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        self._model = AutoModel.from_pretrained(IJEPA_PATH, dtype=dtype, local_files_only=True, cache_dir=DRIVE_HF_CACHE)
        if device == "cuda":
            self._model = self._model.cuda()
        self._model.eval()
        print(f"  ✅ I-JEPA loaded")

    @torch.no_grad()
    def embed(self, image: Image.Image) -> torch.Tensor:
        self._load()
        if image is None:
            return torch.zeros(1, 1280, dtype=torch.float32)
        if image.mode != 'RGB':
            image = image.convert('RGB')
        if self._processor is not None:
            inputs = self._processor(images=image, return_tensors="pt")
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}
        else:
            transform = T.Compose([
                T.Resize((self._SIZE, self._SIZE)),
                T.ToTensor(),
                T.Normalize(mean=self._MEAN, std=self._STD),
            ])
            tensor = transform(image).unsqueeze(0)
            if torch.cuda.is_available():
                tensor = tensor.cuda()
            inputs = {"pixel_values": tensor}
        outputs = self._model(**inputs)
        if hasattr(outputs, 'last_hidden_state'):
            emb = outputs.last_hidden_state[:, 1:, :].mean(dim=1).float()
        else:
            emb = outputs[0].mean(dim=1).float()
        return F.normalize(emb, dim=-1)

class VJEPAExtractor:
    _MEAN = (0.485, 0.456, 0.406)
    _STD = (0.229, 0.224, 0.225)
    _SIZE = 256

    def __init__(self):
        self._model = None
        self._processor = None

    def _load(self):
        if self._model is not None:
            return
        print("  🎬 Loading V-JEPA...")
        try:
            self._processor = AutoVideoProcessor.from_pretrained(VJEPA_PATH, cache_dir=DRIVE_HF_CACHE)
        except Exception:
            self._processor = None
        for attn in [ATTN_IMPL, "sdpa", "eager"]:
            try:
                self._model = AutoModel.from_pretrained(
                    VJEPA_PATH,
                    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                    local_files_only=True,
                    attn_implementation=attn,
                    cache_dir=DRIVE_HF_CACHE,
                )
                if torch.cuda.is_available():
                    self._model = self._model.cuda()
                self._model.eval()
                print(f"  ✅ V-JEPA loaded")
                return
            except Exception:
                continue
        raise RuntimeError("Failed to load V-JEPA")

    @torch.no_grad()
    def embed(self, frames: List[Image.Image]) -> torch.Tensor:
        self._load()
        if not frames:
            return torch.zeros(1, VJEPA_EMBED_DIM, dtype=torch.float32)
        if len(frames) < NUM_FRAMES:
            frames = frames + [frames[-1]] * (NUM_FRAMES - len(frames))
        frames = frames[:NUM_FRAMES]
        if self._processor is not None:
            inputs = self._processor(videos=frames, return_tensors="pt")
            if torch.cuda.is_available():
                inputs = {k: v.cuda() for k, v in inputs.items()}
        else:
            transform = T.Compose([
                T.Resize((self._SIZE, self._SIZE)),
                T.ToTensor(),
                T.Normalize(mean=self._MEAN, std=self._STD),
            ])
            tensors = [transform(f.convert("RGB")) for f in frames]
            video = torch.stack(tensors, dim=0).unsqueeze(0)
            if torch.cuda.is_available():
                video = video.cuda()
            inputs = {"pixel_values_videos": video}
        outputs = self._model(**inputs)
        if hasattr(outputs, 'last_hidden_state'):
            emb = outputs.last_hidden_state.mean(dim=1).float()
        else:
            emb = outputs[0].mean(dim=1).float()
        return F.normalize(emb, dim=-1)

class PerceptualFuser:
    def __init__(self, has_audio: bool = False):
        self._fused_dim = FUSED_DIM_AUDIO if has_audio else FUSED_DIM_VISUAL
        self._proj = torch.nn.Linear(self._fused_dim, VJEPA_EMBED_DIM, bias=False)
        torch.nn.init.xavier_uniform_(self._proj.weight)
        if torch.cuda.is_available():
            self._proj = self._proj.cuda()
        self._proj.eval()

    @torch.no_grad()
    def fuse(self, i_emb: Optional[torch.Tensor], v_emb: Optional[torch.Tensor], a_emb: Optional[torch.Tensor] = None) -> jnp.ndarray:
        if i_emb is None:
            i_emb = torch.zeros(1, 1280, dtype=torch.float32)
        if v_emb is None:
            v_emb = torch.zeros(1, 1024, dtype=torch.float32)
        parts = [i_emb.float(), v_emb.float()]
        if self._fused_dim == FUSED_DIM_AUDIO:
            if a_emb is None:
                a_emb = torch.zeros(1, AUDIO_EMBED_DIM, dtype=torch.float32)
            parts.append(a_emb.float())
        if torch.cuda.is_available():
            parts = [p.cuda() for p in parts]
        fused = torch.cat(parts, dim=-1)
        return jnp.array(F.normalize(self._proj(fused), dim=-1).cpu().numpy())

_ijepa = _vjepa = _fuser = None

def _get_perception(has_audio: bool = False):
    global _ijepa, _vjepa, _fuser
    if _fuser is None:
        _ijepa = IJEPAExtractor()
        _vjepa = VJEPAExtractor()
        _fuser = PerceptualFuser(has_audio=has_audio)
    return _ijepa, _vjepa, _fuser

def extract_perceptual(frames: List = None, image: Image.Image = None, audio_emb: Optional[torch.Tensor] = None, has_audio: bool = False) -> jnp.ndarray:
    if frames is None and image is None:
        return jax.random.normal(_h2e_key, (1, VJEPA_EMBED_DIM))
    ijepa, vjepa, fuser = _get_perception(has_audio=has_audio)
    v_emb = vjepa.embed(frames) if frames else None
    i_emb = ijepa.embed(image) if image else None
    return fuser.fuse(i_emb, v_emb, audio_emb)

# =============================================================================
# DEEPSEEK CLIENT
# =============================================================================
os.environ["LLAMA_CPP_TEMP"] = DRIVE_TMP
os.environ["LLAMA_CPP_CACHE"] = DRIVE_LLAMA_CACHE
LLAMA_TEMP_DIR = os.path.join(DRIVE_TMP, "llama_cpp")
os.makedirs(LLAMA_TEMP_DIR, exist_ok=True)
os.environ["TMPDIR"] = LLAMA_TEMP_DIR
import tempfile
tempfile.tempdir = LLAMA_TEMP_DIR

class DeepSeekClient:
    def __init__(self):
        if not _HAS_LLAMA:
            raise RuntimeError("llama-cpp-python not found")
        gguf = _find_gguf(DS_PATH)
        if gguf is None:
            raise RuntimeError(f"No GGUF found in {DS_PATH}")
        print(f"\n🧠 Loading DeepSeek-R1-Distill-Llama-70B ({_total_size_gb:.1f} GB)")
        print(f"  📁 Model path: {gguf}")
        gpu_layers = 0
        try:
            import subprocess
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.free', '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=5)
            if result.returncode == 0:
                gpu_mem_free = int(result.stdout.strip())
                gpu_layers = min(20, max(0, int(gpu_mem_free / 500)))
                print(f"  🎮 GPU offloading {gpu_layers} layers")
        except:
            pass
        total_bytes = int(_total_size_gb * 1024 ** 3)
        pbar, stop, t_thread = _make_tqdm_loader(total_bytes, "  Loading model")
        try:
            t0 = time.time()
            self._llm = _Llama(model_path=gguf, n_ctx=4096, n_threads=max(1, os.cpu_count() // 2), n_gpu_layers=gpu_layers, n_batch=512, verbose=False, use_mmap=True, use_mlock=False, seed=SEED)
            print(f"\n  ✅ DeepSeek loaded in {time.time() - t0:.1f}s")
        finally:
            stop.set()
            t_thread.join()
            pbar.close()

    def generate(self, prompt: str, max_tokens: int = 512, temp: float = 0.7) -> str:
        out = self._llm(prompt, max_tokens=max_tokens, temperature=temp, echo=False)
        return out["choices"][0]["text"].strip()

    def chat(self, system: str, user: str, max_tokens: int = 512, temp: float = 0.7) -> str:
        prompt = f"<|system|>\n{system}\n<|user|>\n{user}\n<|assistant|>\n"
        return self.generate(prompt, max_tokens, temp)

    def chat_stream(self, system: str, user: str, max_tokens: int = 512, temp: float = 0.7) -> Iterator[str]:
        prompt = f"<|system|>\n{system}\n<|user|>\n{user}\n<|assistant|>\n"
        for chunk in self._llm(prompt, max_tokens=max_tokens, temperature=temp, stream=True, echo=False):
            text = chunk["choices"][0].get("text", "")
            if text:
                yield text

# =============================================================================
# SESSION
# =============================================================================
@dataclass
class SessionState:
    session_id: str
    created_at: str
    last_updated: str
    query_history: list = field(default_factory=list)
    topic_depths: dict = field(default_factory=dict)
    sroi_log: list = field(default_factory=list)
    embedding_log: list = field(default_factory=list)
    total_turns: int = 0
    passed_turns: int = 0

    def to_dict(self) -> dict:
        return {
            "session_id": self.session_id,
            "created_at": self.created_at,
            "last_updated": self.last_updated,
            "query_history": self.query_history,
            "topic_depths": self.topic_depths,
            "sroi_log": self.sroi_log,
            "embedding_log": self.embedding_log,
            "total_turns": self.total_turns,
            "passed_turns": self.passed_turns,
        }

    @classmethod
    def from_dict(cls, d: dict) -> "SessionState":
        return cls(
            session_id=d["session_id"],
            created_at=d["created_at"],
            last_updated=d["last_updated"],
            query_history=d.get("query_history", []),
            topic_depths=d.get("topic_depths", {}),
            sroi_log=d.get("sroi_log", []),
            embedding_log=d.get("embedding_log", []),
            total_turns=d.get("total_turns", 0),
            passed_turns=d.get("passed_turns", 0),
        )

class SessionStore:
    @staticmethod
    def _path(sid: str) -> Path:
        return SESSION_DIR / f"{sid}.json"

    @classmethod
    def new(cls) -> SessionState:
        now = datetime.datetime.now(datetime.timezone.utc).isoformat()
        s = SessionState(session_id=str(uuid.uuid4())[:8], created_at=now, last_updated=now)
        cls.save(s)
        return s

    @classmethod
    def save(cls, s: SessionState) -> None:
        s.last_updated = datetime.datetime.now(datetime.timezone.utc).isoformat()
        p = cls._path(s.session_id)
        tmp = p.with_suffix(".tmp")
        with open(tmp, "w") as f:
            json.dump(s.to_dict(), f, indent=2)
        tmp.replace(p)

# =============================================================================
# TOPIC TRACKER
# =============================================================================
class TopicDepthTracker:
    DEPTH_LABELS = {0: "surface", 1: "surface", 2: "mechanistic", 3: "mechanistic", 4: "principled", 5: "principled", 6: "principled", 7: "expert", 8: "expert", 9: "expert", 10: "expert"}
    JARGON = ["gradient", "latent", "encoder", "manifold", "activation", "loss", "backprop", "attention", "transformer", "embedding"]

    def __init__(self, client: DeepSeekClient, depths: dict):
        self.client = client
        self.depths = depths
        self._topic_cache: dict[str, list] = {}

    def extract_topics(self, query: str) -> List[str]:
        key = query.lower().strip()[:80]
        if key in self._topic_cache:
            return self._topic_cache[key]
        topics = []
        q_lower = query.lower()
        if "neural" in q_lower or "network" in q_lower:
            topics.append("neural_networks")
        if "gradient" in q_lower:
            topics.append("gradient_descent")
        if "latent" in q_lower or "embedding" in q_lower:
            topics.append("latent_space")
        if "encoder" in q_lower:
            topics.append("encoder")
        if "activation" in q_lower or "gelu" in q_lower or "relu" in q_lower:
            topics.append("activations")
        if not topics:
            topics = ["general"]
        self._topic_cache[key] = topics[:3]
        return topics[:3]

    def progression_score(self, query: str, topics: List[str], history: List[str]) -> float:
        if len(history) < 3:
            return 0.65
        topic_known = max((self.depths.get(t, 0) for t in topics), default=0)
        complexity = min(len(query.split()) / 15, 1.0)
        jboost = min(sum(1 for w in self.JARGON if w in query.lower()) * 0.05, 0.20)
        apparent = min(8, int((complexity + jboost) * 8))
        gap = apparent - topic_known
        if gap <= 0:
            return max(0.30, 0.70 - abs(gap) * 0.10)
        elif gap == 1:
            return 0.90
        elif gap == 2:
            return 0.75
        elif gap == 3:
            return 0.55
        else:
            return max(0.20, 0.55 - (gap - 3) * 0.10)

    def advance(self, topics: List[str]) -> None:
        for t in topics:
            self.depths[t] = min(10, self.depths.get(t, 0) + 1)

    def summary(self) -> str:
        if not self.depths:
            return "(no topics covered yet)"
        return "  ".join(f"{t}={v}[{self.DEPTH_LABELS[v]}]" for t, v in sorted(self.depths.items()))

# =============================================================================
# SROI SCORE
# =============================================================================
@dataclass
class SROIScore:
    depth: float
    relevance: float
    novelty: float
    precision: float
    progression: float
    aggregate: float
    passed: bool
    floor_fail: str
    rationale: str
    topics: list = field(default_factory=list)

    def __str__(self) -> str:
        b = lambda v: "█" * int(v * 10) + "░" * (10 - int(v * 10))
        flag = f"  │  ⚠️  Floor fail on '{self.floor_fail}'\n" if self.floor_fail else ""
        return (
            f"\n  ┌─ SROI GATE ──────────────────────┐\n"
            f"  │  Depth       {b(self.depth)}  {self.depth:.2f}\n"
            f"  │  Relevance   {b(self.relevance)}  {self.relevance:.2f}\n"
            f"  │  Novelty     {b(self.novelty)}  {self.novelty:.2f}\n"
            f"  │  Precision   {b(self.precision)}  {self.precision:.2f}\n"
            f"  │  Progression {b(self.progression)}  {self.progression:.2f}\n"
            f"  │  ─────────────────────────────────\n"
            f"  │  Aggregate   {b(self.aggregate)}  {self.aggregate:.2f}"
            f"  {'✅ PASS' if self.passed else '❌ GATE'}\n{flag}"
            f"  │  Topics: {', '.join(self.topics) or 'none'}\n"
            f"  │  Rationale: {self.rationale[:58]}...\n"
            f"  └────────────────────────────────┘"
        )

    def to_log_dict(self) -> dict:
        return {
            "depth": float(self.depth),
            "relevance": float(self.relevance),
            "novelty": float(self.novelty),
            "precision": float(self.precision),
            "progression": float(self.progression),
            "aggregate": float(self.aggregate),
            "passed": str(self.passed),
            "floor_fail": self.floor_fail,
            "rationale": self.rationale,
            "topics": self.topics,
            "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        }

# =============================================================================
# SROI EVALUATOR
# =============================================================================
class SROIEvaluator:
    WEIGHTS = {"depth": 0.35, "relevance": 0.25, "novelty": 0.15, "precision": 0.15, "progression": 0.10}

    def __init__(self, client: DeepSeekClient, tracker: TopicDepthTracker, threshold: float = SROI_THRESHOLD):
        self.client = client
        self.tracker = tracker
        self.threshold = threshold

    @staticmethod
    def _cosine_novelty(z: jnp.ndarray, embedding_log: List[List[float]]) -> float:
        if len(embedding_log) < 2:
            return 0.65
        past = jnp.array(embedding_log[-5:])
        mean_vec = past.mean(axis=0)
        z_flat = z[0]
        cos = float(jnp.dot(z_flat, mean_vec) / (jnp.linalg.norm(z_flat) * jnp.linalg.norm(mean_vec) + 1e-8))
        return float(jnp.clip(1.0 - cos, 0.0, 1.0))

    def score(self, query: str, z: jnp.ndarray, history: List[str], embedding_log: List[List[float]]) -> SROIScore:
        topics = self.tracker.extract_topics(query)
        prog_score = self.tracker.progression_score(query, topics, history)
        novelty_val = self._cosine_novelty(z, embedding_log)

        words = query.lower().split()
        word_count = len(words)
        has_jargon = any(w in query.lower() for w in self.tracker.JARGON)

        if has_jargon and word_count > 15:
            depth = 0.9
        elif has_jargon:
            depth = 0.7
        else:
            depth = 0.5

        relevance = 0.9


        if "?" in query and word_count > 10:
            precision = 0.7
        elif "?" in query:
            precision = 0.5
        else:
            precision = 0.4

        agg = (depth * 0.35 + relevance * 0.25 + novelty_val * 0.15 + precision * 0.15 + prog_score * 0.10)

        floor_fail = ""
        if depth < SROI_FLOOR:
            floor_fail = "depth"
        elif relevance < SROI_FLOOR:
            floor_fail = "relevance"
        elif precision < SROI_FLOOR:
            floor_fail = "precision"

        return SROIScore(
            depth=depth,
            relevance=relevance,
            novelty=novelty_val,
            precision=precision,
            progression=prog_score,
            aggregate=agg,
            passed=(agg >= self.threshold) and not floor_fail,
            floor_fail=floor_fail,
            rationale=f"Query: {query[:50]}...",
            topics=topics,
        )

# =============================================================================
# SCAFFOLDING ENGINE
# =============================================================================
class ScaffoldingEngine:
    def __init__(self, client: DeepSeekClient, tracker: TopicDepthTracker):
        self.client = client
        self.tracker = tracker

    def build_bridge(self, gated_query: str, sroi: SROIScore, history: List[str]) -> Tuple[str, List[str]]:
        known = max((self.tracker.depths.get(t, 0) for t in sroi.topics), default=0)
        if known <= 1:
            questions = [
                f"Can you explain what {sroi.topics[0] if sroi.topics else 'this concept'} means in your own words?",
                "What are the basic components that make this work?",
                "Could you give me a simple example?"
            ]
        elif known <= 3:
            questions = [
                f"How does this relate to the fundamental principles?",
                "What happens if you change one of the key parameters?",
                "Can you walk me through the process step by step?"
            ]
        else:
            questions = [
                "What are the theoretical limitations of this approach?",
                "How would you optimize this for a different scenario?",
                "Can you derive the mathematical formulation?"
            ]
        lines = "\n".join(f"  {i+1}. {q}" for i, q in enumerate(questions[:3]))
        return f"🪜  SCAFFOLD PATH:\n{lines}\n\n  Work through these.", questions

# =============================================================================
# AUDITOR
# =============================================================================
class DeepSeekAuditor:
    def __init__(self, client: DeepSeekClient):
        self.client = client

    def respond(self, z: jnp.ndarray, query: str, sroi: SROIScore, history: List[str], topic_summary: str) -> str:
        system = "You are an expert tutor. Provide a helpful, educational response that probes deeper understanding."
        user = f"Student asked: '{query}'\nCurrent depth: {topic_summary}\nRespond helpfully:"
        return self.client.chat(system, user, 500)

    def respond_stream(self, z: jnp.ndarray, query: str, sroi: SROIScore, history: List[str], topic_summary: str) -> Iterator[str]:
        system = "You are an expert tutor. Provide a helpful, educational response that probes deeper understanding."
        user = f"Student asked: '{query}'\nCurrent depth: {topic_summary}\nRespond helpfully:"
        yield from self.client.chat_stream(system, user, 500)

# =============================================================================
# JAX ENCODER
# =============================================================================
class RMSNorm(nn.Module):
    @nn.compact
    def __call__(self, x: jnp.ndarray) -> jnp.ndarray:
        s = self.param("scale", nn.initializers.ones, (x.shape[-1],))
        return (x / jnp.sqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + 1e-6)) * s

class MultiHeadSelfAttention(nn.Module):
    embed_dim: int
    num_heads: int

    @nn.compact
    def __call__(self, x: jnp.ndarray, deterministic: bool = True) -> jnp.ndarray:
        B, D = x.shape
        head_dim = self.embed_dim // self.num_heads
        qkv = nn.Dense(3 * self.embed_dim, use_bias=False)(x).reshape(B, 3, self.num_heads, head_dim)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        scores = jnp.einsum("bhd,bhd->bh", q, k) / jnp.sqrt(head_dim)
        attn = nn.Dropout(0.1)(jax.nn.softmax(scores), deterministic)
        return nn.Dense(self.embed_dim, use_bias=False)(jnp.einsum("bh,bhd->bd", attn, v))

class EncoderBlock(nn.Module):
    embed_dim: int
    num_heads: int
    ff_mult: int
    dropout: float

    @nn.compact
    def __call__(self, x: jnp.ndarray, deterministic: bool = True) -> jnp.ndarray:
        r = x
        x = RMSNorm()(x)
        x = MultiHeadSelfAttention(self.embed_dim, self.num_heads)(x, deterministic)
        x = nn.Dropout(self.dropout)(x, deterministic) + r
        r = x
        x = RMSNorm()(x)
        x = nn.gelu(nn.Dense(self.embed_dim * self.ff_mult, use_bias=False)(x))
        x = nn.Dropout(self.dropout)(x, deterministic)
        x = nn.Dense(self.embed_dim, use_bias=False)(x)
        return nn.Dropout(self.dropout)(x, deterministic) + r

class SovereignEncoder(nn.Module):
    latent_dim: int = VJEPA_EMBED_DIM
    depth: int = ENCODER_DEPTH
    num_heads: int = ENCODER_HEADS
    ff_mult: int = ENCODER_FF_MULT
    dropout: float = ENCODER_DROPOUT

    @nn.compact
    def __call__(self, x: jnp.ndarray, deterministic: bool = True) -> jnp.ndarray:
        x = RMSNorm()(nn.Dense(self.latent_dim, use_bias=False, kernel_init=nn.initializers.glorot_uniform())(x))
        for _ in range(self.depth):
            x = EncoderBlock(self.latent_dim, self.num_heads, self.ff_mult, self.dropout)(x, deterministic)
        return nn.Dense(self.latent_dim, use_bias=False)(RMSNorm()(x))

# =============================================================================
# ORCHESTRATOR
# =============================================================================
class H2EJEPAOrchestrator:
    def __init__(self, session: SessionState, client: DeepSeekClient):
        self.session = session
        self.client = client
        self.tracker = TopicDepthTracker(client, session.topic_depths)
        self.evaluator = SROIEvaluator(client, self.tracker)
        self.scaffolder = ScaffoldingEngine(client, self.tracker)
        self.auditor = DeepSeekAuditor(client)
        self.encoder = SovereignEncoder()
        self.params = self.encoder.init(_enc_key, jnp.ones((1, VJEPA_EMBED_DIM)))["params"]

    def step(self, query: str, perceptual_emb: jnp.ndarray, stream: bool = False) -> dict:
        z = self.encoder.apply({"params": self.params}, perceptual_emb, deterministic=True)
        self.session.embedding_log.append(z[0].tolist())
        if len(self.session.embedding_log) > 20:
            self.session.embedding_log = self.session.embedding_log[-20:]

        sroi = self.evaluator.score(query, z, self.session.query_history, self.session.embedding_log)

        if sroi.passed:
            if stream:
                response = self.auditor.respond_stream(z, query, sroi, self.session.query_history, self.tracker.summary())
            else:
                response = self.auditor.respond(z, query, sroi, self.session.query_history, self.tracker.summary())
            self.tracker.advance(sroi.topics)
            status = "✅ PASS"
        else:
            scaffold_text, _ = self.scaffolder.build_bridge(query, sroi, self.session.query_history)
            gate_type = f"FLOOR:{sroi.floor_fail.upper()}" if sroi.floor_fail else "AGGREGATE"
            response = f"❌ GATED [{gate_type}]\n{sroi.rationale}\n\n{scaffold_text}"
            status = f"GATED — {gate_type}"

        self.session.query_history.append(query)
        self.session.total_turns += 1
        self.session.passed_turns += int(sroi.passed)
        self.session.sroi_log.append(sroi.to_log_dict())
        SessionStore.save(self.session)

        return {"sroi": sroi, "response": response, "status": status, "topic_summary": self.tracker.summary(), "streamed": stream and sroi.passed}

# =============================================================================
# RUN
# =============================================================================
DEMO_QUERIES = [
    "What is a neural network?",
    "How does latent space topology constrain encoder gradient flow during non-stationary temporal inputs?",
    "Why does an encoder use non-linear activations? What happens if you remove them?",
    "How does GELU's smoothness affect gradient flow compared to ReLU in deep residual encoders?",
]

def _print_result(turn: int, query: str, result: dict) -> None:
    sep = "=" * 68
    print(f"\n{sep}\n  TURN {turn}: {query[:72]}{'...' if len(query)>72 else ''}\n{sep}")
    print(result["sroi"])
    print(f"  Topic Depths : {result['topic_summary']}")
    print(f"  Status       : {result['status']}")
    print(f"\n  Response:\n")
    if result.get("streamed") and hasattr(result["response"], "__iter__") and not isinstance(result["response"], str):
        for chunk in result["response"]:
            print(chunk, end="", flush=True)
        print()
    else:
        for line in str(result["response"]).split("\n"):
            print(f"    {line}")
    print()

def run(use_stream: bool = True, use_audio: bool = True) -> None:
    print(f"🔐 H2E | Seed: {SEED}")
    print(f"👁  I-JEPA:    {IJEPA_PATH}")
    print(f"🎬 V-JEPA:    {VJEPA_PATH}")
    print(f"🧠 DeepSeek:  {DEEPSEEK_GGUF}")
    print(f"📊 SROI:      threshold={SROI_THRESHOLD}  floor={SROI_FLOOR}")
    print(f"🎥 Video:     {VIDEO_PATH}")
    print(f"📁 Sessions:  {SESSION_DIR}\n")

    loader = TartanVideoLoader()
    frames, src = loader.load(VIDEO_PATH, NUM_FRAMES)
    keyframe = loader.extract_keyframe(frames)
    print(f"  📹 {src} | {len(frames)} frames")

    audio_emb, transcript = None, ""
    if use_audio and _HAS_AUDIO and os.path.exists(VIDEO_PATH):
        audio_emb, transcript = loader.extract_audio_embed(VIDEO_PATH)

    has_audio = audio_emb is not None
    perceptual_emb = extract_perceptual(frames, keyframe, audio_emb, has_audio=has_audio)
    print()

    client = DeepSeekClient()
    session = SessionStore.new()
    orch = H2EJEPAOrchestrator(session, client)
    print(f"  🆕 Session: {session.session_id}\n")

    for i, q in enumerate(DEMO_QUERIES, 1):
        r = orch.step(q, perceptual_emb, stream=use_stream)
        _print_result(i, q, r)

    print("=" * 68)
    print(f"  SUMMARY  : {session.total_turns} turns, {session.passed_turns} passed")
    print(f"  Topics   : {orch.tracker.summary()}")
    print(f"  Session  : {SESSION_DIR / session.session_id}.json")
    print("=" * 68)

if __name__ == "__main__":
    run()

✅ Drive redirect confirmed
🔐 H2E | Seed: 123
👁  I-JEPA:    /content/drive/MyDrive/H2E_Models/i_jepa
🎬 V-JEPA:    /content/drive/MyDrive/H2E_Models/v_jepa
🧠 DeepSeek:  /content/drive/MyDrive/H2E_Models/deepseek_70b/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf
📊 SROI:      threshold=0.5  floor=0.15
🎥 Video:     /content/drive/MyDrive/datasets/TartanAviation/vision/1_2023-02-22-15-21-49/1_2023-02-22-15-21-49.mp4
📁 Sessions:  /content/drive/MyDrive/H2E_Models/sessions

  🎬 Loaded 16 frames
  📹 /content/drive/MyDrive/datasets/TartanAviation/vision/1_2023-02-22-15-21-49/1_2023-02-22-15-21-49.mp4 | 16 frames
  🎬 Loading V-JEPA...


Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

  ✅ V-JEPA loaded
  📷 Loading I-JEPA...


The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

  ✅ I-JEPA loaded


🧠 Loading DeepSeek-R1-Distill-Llama-70B (39.6 GB)
  📁 Model path: /content/drive/MyDrive/H2E_Models/deepseek_70b/DeepSeek-R1-Distill-Llama-70B-Q4_K_M.gguf
  🎮 GPU offloading 20 layers


  Loading model:   0%|          | 0.00/42.5G [00:00<?, ?B/s]

llama_context: n_ctx_seq (4096) < n_ctx_train (131072) -- the full capacity of the model will not be utilized



  ✅ DeepSeek loaded in 374.2s
  🆕 Session: 5ed56327


  TURN 1: What is a neural network?

  ┌─ SROI GATE ──────────────────────┐
  │  Depth       █████░░░░░  0.50
  │  Relevance   █████████░  0.90
  │  Novelty     ██████░░░░  0.65
  │  Precision   █████░░░░░  0.50
  │  Progression ██████░░░░  0.65
  │  ─────────────────────────────────
  │  Aggregate   ██████░░░░  0.64  ✅ PASS
  │  Topics: neural_networks
  │  Rationale: Query: What is a neural network?......
  └────────────────────────────────┘
  Topic Depths : neural_networks=1[surface]
  Status       : ✅ PASS

  Response:

A neural network is a machine learning model inspired by the structure and function of the human brain. It consists of layers of interconnected nodes or "neurons," which process and transmit information. These networks are trained on data to learn patterns and make predictions or decisions. Neural networks are fundamental to deep learning and are used in applications like image recognition, natural language proc